In [ ]:
# @title 0) Colab 커널 bootstrap: 저장소 clone/update + 최소 설치
import os
import subprocess
import sys
from pathlib import Path

print("[bootstrap] start", flush=True)

REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
REPO_REF = os.environ.get("MINDSCOPEX_REPO_REF", "codex/reasoning-output-analysis").strip() or "main"
TARGET = Path("/content/colab")
MARKER = Path("src") / "mindscopex_analysis" / "__init__.py"
TRANSFORMERS_PROFILE = os.environ.get("CARWASH_TRANSFORMERS_PROFILE", "qwen35").strip().lower()
# Profiles: qwen35 = install HF transformers main; legacy_ouro = pin transformers 4.54.1; none = leave env as-is.
os.environ["CARWASH_TRANSFORMERS_PROFILE"] = TRANSFORMERS_PROFILE


def run(cmd, cwd=None, timeout=300, required=True):
    print("+", " ".join(map(str, cmd)), flush=True)
    try:
        subprocess.run(
            cmd,
            cwd=str(cwd) if cwd else None,
            timeout=timeout,
            check=True,
        )
    except Exception as exc:
        print(f"[bootstrap] command failed: {exc}", flush=True)
        if required:
            raise


if Path("/content").exists():
    if (TARGET / ".git").exists():
        run(["git", "fetch", "origin", REPO_REF], cwd=TARGET, timeout=90, required=False)
        run(["git", "checkout", "-B", REPO_REF, f"origin/{REPO_REF}"], cwd=TARGET, timeout=60, required=False)
        run(["git", "pull", "--ff-only", "origin", REPO_REF], cwd=TARGET, timeout=90, required=False)
    elif not TARGET.exists():
        run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(TARGET)], timeout=180)
    elif not (TARGET / MARKER).is_file():
        alt = Path("/content/mindscopex_analysis")
        if (alt / ".git").exists():
            run(["git", "fetch", "origin", REPO_REF], cwd=alt, timeout=90, required=False)
            run(["git", "checkout", "-B", REPO_REF, f"origin/{REPO_REF}"], cwd=alt, timeout=60, required=False)
            run(["git", "pull", "--ff-only", "origin", REPO_REF], cwd=alt, timeout=90, required=False)
        elif not alt.exists():
            run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(alt)], timeout=180)
        TARGET = alt
    if not (TARGET / MARKER).is_file():
        raise FileNotFoundError(f"저장소 marker를 찾지 못했습니다: {TARGET / MARKER}")
    os.chdir(TARGET)
    os.environ["MINDSCOPEX_ROOT"] = str(TARGET)
    print("cwd =", Path.cwd(), flush=True)
    print("REPO_REF =", REPO_REF, flush=True)
    print("MINDSCOPEX_ROOT =", os.environ["MINDSCOPEX_ROOT"], flush=True)
    run([sys.executable, "-m", "pip", "install", "-e", "."], timeout=600)
    print("TRANSFORMERS_PROFILE =", TRANSFORMERS_PROFILE, flush=True)
    if TRANSFORMERS_PROFILE == "qwen35":
        run([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-U",
            "transformers @ git+https://github.com/huggingface/transformers.git@main",
            "accelerate",
            "pillow",
            "torchvision",
        ], timeout=1200)
    elif TRANSFORMERS_PROFILE == "legacy_ouro":
        run([sys.executable, "-m", "pip", "install", "transformers==4.54.1"], timeout=600)
    elif TRANSFORMERS_PROFILE in {"none", "manual"}:
        print("[bootstrap] leaving transformers unchanged", flush=True)
    else:
        raise ValueError("Unknown CARWASH_TRANSFORMERS_PROFILE: " + TRANSFORMERS_PROFILE)
    if "transformers" in sys.modules:
        raise SystemExit("Transformers was already imported. Restart the Colab runtime, then rerun from the first cell.")
    print("[bootstrap] done", flush=True)
else:
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / MARKER).is_file():
            os.environ["MINDSCOPEX_ROOT"] = str(base)
            print("로컬 저장소에서 실행 중입니다:", base, flush=True)
            break
    else:
        print("저장소 루트를 찾지 못했습니다. Colab 커널이면 이 셀을 맨 먼저 다시 실행하세요.", flush=True)


# 세차장 50m 문제: Qwen thinking/non-thinking + looped transformer 출력 비교

문제:

> 나는 세차를 하고 싶다. 세차장은 50m 앞에 있다. 걸어가야 할까, 차로 가야 할까?

정답을 하나로 강제하기보다, 모델이 어떤 상황 해석을 하는지 봅니다. 핵심 관찰 포인트는 다음입니다.

- 세차 대상이 `차`라는 목표를 유지하는가?
- 50m라는 짧은 거리 때문에 `걸어가라`는 표면 휴리스틱에 끌리는가?
- “몸만 가는 것”과 “차를 세차장에 가져가는 것”을 구분하는가?
- 애매함을 인식하고 조건부 답을 하는가?
- 최종 답은 맞아도, `think` 안에서 문제를 몸만 50m 이동하는 문제로 잘못 해석하지 않는가?

Qwen 모델은 `think_off`/`think_on`을 비교하고, looped transformer 계열은 Ouro/Parcae 후보를 순차 실행합니다. 결과 테이블은 `thinking`과 `answer`를 별도 컬럼으로 저장하고, 각각의 휴리스틱 판정과 정렬 여부를 따로 보여줍니다.

Colab 기본 실행은 `CARWASH_TRANSFORMERS_PROFILE=qwen35`입니다. Qwen3.5는 Hugging Face `transformers` main이 필요합니다. 현재 Ouro 카드는 `trust_remote_code=True`로 직접 로드하는 방식을 안내하므로 같은 런타임에서 함께 시도하되, Ouro 쪽 커스텀 코드 문제가 나면 별도 런타임에서 legacy pin을 테스트하세요.


## 1. Import와 설정


In [ ]:
import gc
import os
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoProcessor,
    AutoTokenizer,
    set_seed,
)
import transformers
try:
    from transformers import AutoModelForImageTextToText
except ImportError:
    AutoModelForImageTextToText = None

_REPO_MARK = Path("src") / "mindscopex_analysis" / "__init__.py"


def _find_repo_root() -> Path:
    env = os.environ.get("MINDSCOPEX_ROOT", "").strip()
    if env:
        root = Path(env).expanduser().resolve()
        if (root / _REPO_MARK).is_file():
            return root
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / _REPO_MARK).is_file():
            return base
    raise FileNotFoundError("src/mindscopex_analysis 를 찾지 못했습니다.")


ROOT = _find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from mindscopex_analysis.notebook_utils import dtype_from_str
from mindscopex_analysis.qwen_scope import format_qwen_chat, split_generation_parts

set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = dtype_from_str("bfloat16" if DEVICE == "cuda" else "float32")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")
OUT_DIR = ROOT / "outputs" / "carwash_model_compare" / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT =", ROOT)
print("torch =", torch.__version__, "cuda =", torch.cuda.is_available(), "device =", DEVICE)
print("transformers =", transformers.__version__)
print("AutoModelForImageTextToText =", AutoModelForImageTextToText is not None)
print("out =", OUT_DIR)


## 2. 문제 변형과 평가 휴리스틱


In [ ]:
CARWASH_KO = "나는 세차를 하고 싶다. 세차장은 50m 앞에 있다. 걸어가야 할까, 차로 가야 할까?"

PROMPTS = [
    {
        "prompt_id": "ko_direct",
        "language": "ko",
        "prompt": CARWASH_KO,
    },
    {
        "prompt_id": "ko_short_answer",
        "language": "ko",
        "prompt": CARWASH_KO + "\n한 문장으로 답하고, 이유를 짧게 말해줘.",
    },
    {
        "prompt_id": "ko_careful",
        "language": "ko",
        "prompt": CARWASH_KO + "\n상황의 함정을 조심해서 판단해줘.",
    },
    {
        "prompt_id": "en_direct",
        "language": "en",
        "prompt": (
            "I want to wash my car. The car wash is 50 meters ahead. "
            "Should I walk or drive?"
        ),
    },
]


ANSWER_OK_LABELS = {"drive", "conditional_or_both"}


def classify_carwash_answer(text: str) -> dict:
    t = text.lower()
    drive_words = [
        "차로",
        "차로 가",
        "운전",
        "차를 타",
        "차를 가져",
        "차를 가져가",
        "차를 가지고",
        "차를 몰",
        "drive",
        "drive there",
        "take the car",
        "bring the car",
    ]
    walk_words = ["걸어", "걸어가", "walk", "on foot"]
    car_target_words = ["세차", "세차장", "차를 씻", "차를 닦", "wash my car", "wash the car", "car wash"]
    transport_task_words = [
        "차를 세차장",
        "차를 가져",
        "차를 가져가",
        "차를 가지고",
        "차를 몰",
        "차가 세차장",
        "bring the car",
        "take the car",
        "drive it",
        "get the car there",
        "car needs to be",
    ]
    ambiguity_words = ["만약", "라면", "depends", "if ", "unless", "애매", "상황"]
    distance_words = ["50m", "50 m", "50미터", "50 meters", "50-meter"]

    keeps_car_target = any(w in t for w in car_target_words)
    distinguishes_transport_task = any(w in t for w in transport_task_words)
    says_drive = any(w in t for w in drive_words) or distinguishes_transport_task
    says_walk = any(w in t for w in walk_words)
    notes_ambiguity = any(w in t for w in ambiguity_words)
    mentions_distance = any(w in t for w in distance_words)

    if says_drive and not says_walk:
        label = "drive"
    elif says_walk and not says_drive:
        label = "walk"
    elif says_drive and says_walk:
        label = "conditional_or_both"
    else:
        label = "unclear"

    return {
        "label": label,
        "says_drive": says_drive,
        "says_walk": says_walk,
        "keeps_car_target": keeps_car_target,
        "distinguishes_transport_task": distinguishes_transport_task,
        "notes_ambiguity": notes_ambiguity,
        "mentions_distance": mentions_distance,
    }


def prefixed_judgment(judged: dict, prefix: str) -> dict:
    return {f"{prefix}_{k}": v for k, v in judged.items()}


def carwash_answer_looks_right(answer_judged: dict) -> bool:
    # 최종 답은 우선 차를 가져가야 한다는 결론을 냈는지로 봅니다.
    return answer_judged["label"] in ANSWER_OK_LABELS


def carwash_thinking_looks_right(thinking: str, thinking_judged: dict):
    if not thinking.strip():
        return None
    return (
        thinking_judged["label"] in ANSWER_OK_LABELS
        and (thinking_judged["keeps_car_target"] or thinking_judged["distinguishes_transport_task"])
    )


def diagnose_thinking(thinking: str, thinking_judged: dict) -> str:
    if not thinking.strip():
        return "no_visible_thinking"
    if carwash_thinking_looks_right(thinking, thinking_judged):
        return "supports_drive_to_wash"
    if thinking_judged["label"] == "walk":
        return "walk_only_trap"
    if not thinking_judged["keeps_car_target"] and not thinking_judged["distinguishes_transport_task"]:
        return "missing_car_target"
    if thinking_judged["label"] == "unclear":
        return "unclear_reasoning"
    return "mixed_or_conflicting_reasoning"


def thinking_answer_alignment(answer_judged: dict, thinking_judged: dict, has_thinking: bool) -> str:
    if not has_thinking:
        return "no_visible_thinking"
    answer_label = answer_judged["label"]
    thinking_label = thinking_judged["label"]
    if answer_label == thinking_label:
        return "same_label"
    if answer_label in ANSWER_OK_LABELS and thinking_label in ANSWER_OK_LABELS:
        return "same_drive_family"
    if answer_label in ANSWER_OK_LABELS and thinking_label == "walk":
        return "answer_drive_thinking_walk"
    if answer_label == "walk" and thinking_label in ANSWER_OK_LABELS:
        return "answer_walk_thinking_drive"
    if "unclear" in {answer_label, thinking_label}:
        return "one_side_unclear"
    return "different_labels"


pd.DataFrame(PROMPTS).pipe(display)


## 3. 모델군 설정


In [ ]:
MODEL_SPECS = [
    # 요청 범위: Qwen3.5 2B/9B/27B/35B + Ouro. Qwen3 1.7B는 baseline에서 제외.
    {
        "tag": "qwen35_2b",
        "family": "qwen",
        "model_id": "Qwen/Qwen3.5-2B",
        "enabled": True,
        "runner": "processor",
        "modes": ["think_off", "think_on"],
    },
    {
        "tag": "qwen35_9b",
        "family": "qwen",
        "model_id": "Qwen/Qwen3.5-9B",
        "enabled": True,
        "runner": "processor",
        "modes": ["think_off", "think_on"],
    },
    {
        "tag": "qwen35_27b",
        "family": "qwen",
        "model_id": "Qwen/Qwen3.5-27B",
        "enabled": True,
        "runner": "processor",
        "modes": ["think_off", "think_on"],
    },
    {
        "tag": "qwen35_35b_a3b",
        "family": "qwen",
        "model_id": "Qwen/Qwen3.5-35B-A3B",
        "enabled": True,
        "runner": "processor",
        "modes": ["think_off", "think_on"],
    },
    # Looped transformer / recurrent-depth 계열.
    {
        "tag": "ouro_1p4b_looplm",
        "family": "looped_transformer",
        "model_id": "ByteDance/Ouro-1.4B",
        "enabled": True,
        "runner": "ouro",
        "modes": ["plain"],
        "note": "LoopLM. trust_remote_code 필요. transformers 호환성 이슈가 있으면 자동 skip됩니다.",
    },
    # 비교용 optional baseline. 필요하면 켜세요.
    {
        "tag": "qwen3_1p7b_optional",
        "family": "qwen",
        "model_id": "Qwen/Qwen3-1.7B",
        "enabled": False,
        "runner": "causal_lm",
        "modes": ["think_off", "think_on"],
    },
    {
        "tag": "parcae_770m_optional",
        "family": "looped_transformer",
        "model_id": "SandyResearch/parcae-770m",
        "enabled": False,
        "modes": ["plain"],
        "runner": "parcae",
        "note": "pip install parcae-lm 후 실행되는 optional runner입니다.",
    },
]

GEN_CONFIGS = [
    {"decode": "greedy", "do_sample": False, "temperature": None, "top_p": None, "n": 1},
    {"decode": "sample", "do_sample": True, "temperature": 0.7, "top_p": 0.9, "n": 3},
]

MAX_NEW_TOKENS = 512
RUN_ONLY_FIRST_N_MODELS = None  # smoke test는 1 또는 2로 지정
RUN_LOOPED_MODELS = True  # 5번 노트북 기본값: enabled=True인 Qwen3.5 전체 + Ouro까지 확인합니다.
INSTALL_PARCAE = False

RUNTIME_PROFILE = os.environ.get("CARWASH_TRANSFORMERS_PROFILE", "qwen35").strip().lower()


def transformers_has_arch(model_type: str) -> bool:
    try:
        from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    except Exception:
        return False
    return model_type in CONFIG_MAPPING_NAMES


if RUNTIME_PROFILE == "ouro":
    # Convenience mode for running only Ouro in a fresh runtime.
    for spec in MODEL_SPECS:
        spec["enabled"] = spec.get("runner") == "ouro"
elif RUNTIME_PROFILE == "legacy_ouro":
    # Last-resort compatibility mode. This disables Qwen3.5 because qwen3_5 needs newer transformers.
    for spec in MODEL_SPECS:
        spec["enabled"] = spec.get("runner") == "ouro"

print("transformers =", transformers.__version__)
print("runtime profile =", RUNTIME_PROFILE)
print("qwen3_5 arch support =", transformers_has_arch("qwen3_5"))
if any("Qwen3.5" in s.get("model_id", "") and s.get("enabled") for s in MODEL_SPECS):
    if AutoModelForImageTextToText is None:
        raise RuntimeError(
            "AutoModelForImageTextToText is unavailable in this transformers build. "
            "Run the first bootstrap cell with CARWASH_TRANSFORMERS_PROFILE=qwen35, "
            "then restart the Colab runtime if transformers had already been imported."
        )
    if not transformers_has_arch("qwen3_5"):
        raise RuntimeError(
            "This transformers build does not support model_type=qwen3_5. "
            "Run the first bootstrap cell with CARWASH_TRANSFORMERS_PROFILE=qwen35, "
            "then restart the Colab runtime if transformers had already been imported."
        )
if any(s.get("runner") == "ouro" and s.get("enabled") for s in MODEL_SPECS):
    print("Ouro enabled: using trust_remote_code=True with the current transformers build.")

pd.DataFrame(MODEL_SPECS).pipe(display)


## 4. 실행 함수


### Optional: legacy transformers pin for Ouro

Current Ouro model card examples use normal `transformers` loading with `trust_remote_code=True`. Keep this cell off unless Ouro custom code fails in the current environment. If you enable the legacy pin, run Ouro in a fresh runtime because Qwen3.5 needs a newer/main Transformers build.


In [ ]:
PIN_TRANSFORMERS_FOR_LEGACY_OURO = False
if PIN_TRANSFORMERS_FOR_LEGACY_OURO:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers==4.54.1"])
    raise SystemExit("Pinned legacy transformers for Ouro. Restart the Colab runtime, then rerun with only Ouro enabled.")


In [ ]:
def _model_kwargs(extra: dict | None = None) -> dict:
    kwargs = {
        "torch_dtype": DTYPE,
        "trust_remote_code": True,
        "low_cpu_mem_usage": True,
    }
    if DEVICE == "cuda":
        kwargs["device_map"] = "auto"
    if extra:
        kwargs.update(extra)
    return kwargs


def _move_inputs_to_device(inputs):
    if hasattr(inputs, "to"):
        return inputs.to(DEVICE)
    return {k: v.to(DEVICE) if hasattr(v, "to") else v for k, v in inputs.items()}


def _safe_decode(decoder, token_ids):
    try:
        return decoder.decode(token_ids, skip_special_tokens=False).strip()
    except TypeError:
        return decoder.decode(token_ids).strip()


def enable_thinking_for_mode(mode: str):
    if mode == "think_on":
        return True
    if mode == "think_off":
        return False
    return None


def format_for_model(tokenizer, prompt: str, mode: str, family: str) -> str:
    if family == "qwen":
        enable_thinking = enable_thinking_for_mode(mode)
        return format_qwen_chat(
            tokenizer,
            prompt,
            system_prompt="You are a careful assistant. Answer the user's question directly.",
            enable_thinking=enable_thinking,
        )

    return (
        "Question:\n"
        f"{prompt}\n\n"
        "Answer directly, then give a brief reason:\n"
    )


def qwen_content(prompt: str) -> list[dict]:
    # Qwen3.5 uses the image-text-to-text processor path, but text-only content is valid.
    return [{"type": "text", "text": prompt}]


def processor_messages(prompt: str, mode: str) -> list[dict]:
    suffix = ""
    if mode == "think_off":
        suffix = "\n\nDo not use extended thinking. Give the direct answer first."
    elif mode == "think_on":
        suffix = "\n\nThink carefully before answering."
    return [
        {
            "role": "system",
            "content": qwen_content("You are a careful assistant. Answer the user's question directly."),
        },
        {"role": "user", "content": qwen_content(prompt + suffix)},
    ]


def append_row(rows, spec, mode, prompt_spec, gen_cfg, sample_idx, raw):
    parts = split_generation_parts(raw)
    thinking = parts.thinking
    answer = parts.answer
    answer_judged = classify_carwash_answer(answer)
    thinking_judged = classify_carwash_answer(thinking)
    has_thinking = bool(thinking.strip())
    has_answer = bool(answer.strip())
    thinking_looks_right = carwash_thinking_looks_right(thinking, thinking_judged)
    rows.append({
        "tag": spec["tag"],
        "family": spec["family"],
        "model_id": spec["model_id"],
        "mode": mode,
        "decode": gen_cfg["decode"],
        "prompt_id": prompt_spec["prompt_id"],
        "language": prompt_spec["language"],
        "sample_idx": sample_idx,
        "split_method": parts.split_method,
        "has_thinking": has_thinking,
        "has_answer": has_answer,
        "thinking": thinking,
        "thinking_words": len(thinking.split()),
        "thinking_chars": len(thinking),
        "answer": answer,
        "answer_words": len(answer.split()),
        "answer_chars": len(answer),
        "thinking_excerpt": thinking[:700],
        "raw_generation": raw,
        "answer_looks_right": carwash_answer_looks_right(answer_judged),
        "thinking_looks_right": thinking_looks_right,
        "thinking_issue": diagnose_thinking(thinking, thinking_judged),
        "think_answer_alignment": thinking_answer_alignment(answer_judged, thinking_judged, has_thinking),
        # Backward-compatible answer judgment columns.
        "label": answer_judged["label"],
        "says_drive": answer_judged["says_drive"],
        "says_walk": answer_judged["says_walk"],
        "keeps_car_target": answer_judged["keeps_car_target"],
        "distinguishes_transport_task": answer_judged["distinguishes_transport_task"],
        "notes_ambiguity": answer_judged["notes_ambiguity"],
        "mentions_distance": answer_judged["mentions_distance"],
        **prefixed_judgment(answer_judged, "answer"),
        **prefixed_judgment(thinking_judged, "thinking"),
    })


def generate_with_causal_lm(spec: dict) -> list[dict]:
    tag = spec["tag"]
    model_id = spec["model_id"]
    family = spec["family"]
    print(f"\n===== loading {tag}: {model_id} (causal_lm) =====", flush=True)

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, **_model_kwargs(spec.get("model_kwargs"))).eval()
    if DEVICE != "cuda":
        model = model.to(DEVICE)
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    rows = []
    try:
        for mode in spec.get("modes", ["plain"]):
            for prompt_spec in PROMPTS:
                formatted = format_for_model(tokenizer, prompt_spec["prompt"], mode, family)
                inputs = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=2048)
                inputs = _move_inputs_to_device(inputs)
                for gen_cfg in GEN_CONFIGS:
                    for sample_idx in range(int(gen_cfg["n"])):
                        kwargs = {
                            "max_new_tokens": MAX_NEW_TOKENS,
                            "do_sample": bool(gen_cfg["do_sample"]),
                            "pad_token_id": tokenizer.pad_token_id,
                            "eos_token_id": tokenizer.eos_token_id,
                        }
                        if gen_cfg["do_sample"]:
                            kwargs.update({
                                "temperature": float(gen_cfg["temperature"]),
                                "top_p": float(gen_cfg["top_p"]),
                                "top_k": 20,
                            })
                        with torch.no_grad():
                            out = model.generate(**inputs, **kwargs)
                        new_tokens = out[0, inputs["input_ids"].shape[1]:]
                        raw = _safe_decode(tokenizer, new_tokens)
                        append_row(rows, spec, mode, prompt_spec, gen_cfg, sample_idx, raw)
    finally:
        del model, tokenizer
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return rows


def generate_with_processor_model(spec: dict) -> list[dict]:
    tag = spec["tag"]
    model_id = spec["model_id"]
    print(f"\n===== loading {tag}: {model_id} (processor image-text-to-text) =====", flush=True)
    if AutoModelForImageTextToText is None:
        raise RuntimeError("AutoModelForImageTextToText is unavailable. Run the bootstrap cell and restart the runtime.")

    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        **_model_kwargs(spec.get("model_kwargs")),
    ).eval()
    if DEVICE != "cuda":
        model = model.to(DEVICE)

    rows = []
    try:
        for mode in spec.get("modes", ["plain"]):
            for prompt_spec in PROMPTS:
                messages = processor_messages(prompt_spec["prompt"], mode)
                chat_kwargs = {
                    "add_generation_prompt": True,
                    "tokenize": True,
                    "return_dict": True,
                    "return_tensors": "pt",
                }
                enable_thinking = enable_thinking_for_mode(mode)
                if enable_thinking is not None:
                    chat_kwargs["enable_thinking"] = enable_thinking
                try:
                    inputs = processor.apply_chat_template(messages, **chat_kwargs)
                except TypeError:
                    # Older processor fallback: build a text-only chat string and pass it back to the processor.
                    chat_kwargs.pop("enable_thinking", None)
                    chat_text = processor.apply_chat_template(
                        messages,
                        add_generation_prompt=True,
                        tokenize=False,
                    )
                    inputs = processor(text=[chat_text], return_tensors="pt")
                inputs = _move_inputs_to_device(inputs)
                input_len = inputs["input_ids"].shape[-1]

                for gen_cfg in GEN_CONFIGS:
                    for sample_idx in range(int(gen_cfg["n"])):
                        kwargs = {
                            "max_new_tokens": MAX_NEW_TOKENS,
                            "do_sample": bool(gen_cfg["do_sample"]),
                        }
                        if gen_cfg["do_sample"]:
                            kwargs.update({
                                "temperature": float(gen_cfg["temperature"]),
                                "top_p": float(gen_cfg["top_p"]),
                                "top_k": 20,
                            })
                        with torch.no_grad():
                            out = model.generate(**inputs, **kwargs)
                        new_tokens = out[0, input_len:]
                        raw = _safe_decode(processor, new_tokens)
                        append_row(rows, spec, mode, prompt_spec, gen_cfg, sample_idx, raw)
    finally:
        del model, processor
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return rows


def generate_with_ouro(spec: dict) -> list[dict]:
    # Ouro uses custom code via trust_remote_code=True; current model card shows standard HF loading.
    return generate_with_causal_lm(spec)


def generate_with_parcae(spec: dict) -> list[dict]:
    if not INSTALL_PARCAE:
        raise RuntimeError("INSTALL_PARCAE=False. Set it to True to run Parcae.")
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "parcae-lm"])
    import parcae_lm

    model_id = spec["model_id"]
    tag = spec["tag"]
    print(f"\n===== loading {tag}: {model_id} =====", flush=True)
    model = parcae_lm.from_pretrained(model_id)
    if hasattr(model, "to"):
        model = model.to(DEVICE)
    rows = []
    try:
        for prompt_spec in PROMPTS:
            prompt = (
                "Question:\n"
                f"{prompt_spec['prompt']}\n\n"
                "Answer directly, then give a brief reason:\n"
            )
            if hasattr(model, "generate"):
                out = model.generate(prompt, max_new_tokens=MAX_NEW_TOKENS)
            else:
                raise RuntimeError("Parcae model object has no generate method in this environment.")
            raw = str(out)
            append_row(
                rows,
                spec,
                "plain",
                prompt_spec,
                {"decode": "default"},
                0,
                raw,
            )
    finally:
        del model
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return rows


## 5. 모델별 순차 실행


In [ ]:
specs = [s for s in MODEL_SPECS if s.get("enabled", False)]
if not RUN_LOOPED_MODELS:
    specs = [s for s in specs if s["family"] != "looped_transformer"]
if RUN_ONLY_FIRST_N_MODELS is not None:
    specs = specs[: int(RUN_ONLY_FIRST_N_MODELS)]

RESULT_COLUMNS = [
    "tag", "family", "model_id", "mode", "decode", "prompt_id", "language", "sample_idx",
    "split_method", "has_thinking", "has_answer",
    "label", "says_drive", "says_walk", "keeps_car_target", "distinguishes_transport_task",
    "notes_ambiguity", "mentions_distance",
    "answer_label", "answer_says_drive", "answer_says_walk", "answer_keeps_car_target",
    "answer_distinguishes_transport_task", "answer_notes_ambiguity", "answer_mentions_distance",
    "thinking_label", "thinking_says_drive", "thinking_says_walk", "thinking_keeps_car_target",
    "thinking_distinguishes_transport_task", "thinking_notes_ambiguity", "thinking_mentions_distance",
    "answer_looks_right", "thinking_looks_right", "thinking_issue", "think_answer_alignment",
    "thinking_words", "thinking_chars", "answer_words", "answer_chars",
    "thinking", "answer", "thinking_excerpt", "raw_generation",
]
FAILURE_COLUMNS = ["tag", "model_id", "family", "runner", "error_type", "error"]

all_rows = []
failures = []
for spec in specs:
    try:
        runner = spec.get("runner", "causal_lm")
        if runner == "parcae":
            rows = generate_with_parcae(spec)
        elif runner == "processor":
            rows = generate_with_processor_model(spec)
        elif runner == "ouro":
            rows = generate_with_ouro(spec)
        else:
            rows = generate_with_causal_lm(spec)
        all_rows.extend(rows)
    except Exception as exc:
        print(f"[FAIL] {spec['tag']}: {type(exc).__name__}: {exc}")
        failures.append({
            "tag": spec["tag"],
            "model_id": spec["model_id"],
            "family": spec["family"],
            "runner": spec.get("runner", "causal_lm"),
            "error_type": type(exc).__name__,
            "error": str(exc),
        })
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

results = pd.DataFrame(all_rows, columns=RESULT_COLUMNS)
failures_df = pd.DataFrame(failures, columns=FAILURE_COLUMNS)

results.to_csv(OUT_DIR / "carwash_model_outputs.csv", index=False, encoding="utf-8-sig")
failures_df.to_csv(OUT_DIR / "carwash_model_failures.csv", index=False, encoding="utf-8-sig")

if not failures_df.empty:
    print("Failures:")
    display(failures_df)

if results.empty:
    raise RuntimeError(
        "No successful model outputs. Check carwash_model_failures.csv or failures_df above. "
        "For Qwen3.5, run the bootstrap cell with CARWASH_TRANSFORMERS_PROFILE=qwen35. "
        "For Ouro custom-code errors, try a fresh runtime or the legacy Ouro pin cell."
    )

display(results[[
    "tag", "family", "mode", "decode", "prompt_id", "sample_idx",
    "split_method", "answer_label", "answer_looks_right", "answer_keeps_car_target",
    "thinking_label", "thinking_looks_right", "thinking_issue", "think_answer_alignment",
    "thinking_words", "answer",
]])


## 6. 요약과 전문 출력


In [ ]:
if results.empty:
    raise ValueError("성공한 모델 출력이 없습니다. failures_df를 확인하세요.")

summary = (
    results
    .groupby(["tag", "family", "mode", "decode", "answer_label"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(["family", "tag", "mode", "decode", "answer_label"])
)
display(summary)
summary.to_csv(OUT_DIR / "carwash_summary.csv", index=False, encoding="utf-8-sig")

thinking_summary = (
    results
    .groupby(["tag", "family", "mode", "thinking_issue", "think_answer_alignment"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(["family", "tag", "mode", "thinking_issue", "think_answer_alignment"])
)
display(thinking_summary)
thinking_summary.to_csv(OUT_DIR / "carwash_thinking_summary.csv", index=False, encoding="utf-8-sig")

judge_summary = (
    results
    .groupby(["tag", "family", "mode"], dropna=False)
    .agg(
        n=("answer", "size"),
        answer_ok_rate=("answer_looks_right", "mean"),
        answer_drive_rate=("answer_says_drive", "mean"),
        answer_walk_rate=("answer_says_walk", "mean"),
        answer_car_target_rate=("answer_keeps_car_target", "mean"),
        answer_transport_task_rate=("answer_distinguishes_transport_task", "mean"),
        answer_ambiguity_rate=("answer_notes_ambiguity", "mean"),
        visible_thinking_rate=("has_thinking", "mean"),
        thinking_ok_rate=("thinking_looks_right", "mean"),
        thinking_car_target_rate=("thinking_keeps_car_target", "mean"),
        thinking_transport_task_rate=("thinking_distinguishes_transport_task", "mean"),
        avg_thinking_words=("thinking_words", "mean"),
    )
    .reset_index()
)
display(judge_summary)
judge_summary.to_csv(OUT_DIR / "carwash_judge_summary.csv", index=False, encoding="utf-8-sig")

diagnostic_cols = [
    "tag", "family", "mode", "decode", "prompt_id", "sample_idx", "split_method",
    "answer_label", "answer_looks_right", "answer_keeps_car_target",
    "thinking_label", "thinking_looks_right", "thinking_issue", "think_answer_alignment",
    "thinking_words", "answer", "thinking",
]
diagnostics = results[diagnostic_cols].copy()
display(diagnostics)
diagnostics.to_csv(OUT_DIR / "carwash_reasoning_diagnostics.csv", index=False, encoding="utf-8-sig")


In [ ]:
PRINT_FULL_THINKING = True

for r in results.itertuples():
    print("=" * 100)
    print(
        f"{r.tag} | {r.family} | {r.mode} | {r.decode} | {r.prompt_id} | sample={r.sample_idx} "
        f"| answer={r.answer_label} | thinking_issue={r.thinking_issue} | alignment={r.think_answer_alignment}"
    )
    print(f"split_method={r.split_method} | has_thinking={r.has_thinking} | has_answer={r.has_answer}")
    if r.has_thinking:
        print("\n[thinking]")
        print(r.thinking if PRINT_FULL_THINKING else r.thinking_excerpt)
    print("\n[answer]")
    print(r.answer)


## 7. 해석 메모

이 문제에서 내가 기대하는 강한 답은 보통 `차로 가야 한다`입니다. 이유는 사용자가 원하는 것은 본인이 50m 이동하는 것이 아니라 `차를 세차장에 가져가 세차하는 것`이기 때문입니다.

다만 더 좋은 답은 조건부일 수 있습니다.

- 자동 세차장/드라이브스루라면 차로 간다.
- 세차장에 사람이 걸어가서 예약만 하거나 물품을 사러 가는 상황이면 걸어갈 수 있다.
- “세차를 하고 싶다”가 “내가 직접 세차 노동을 하고 싶다”라는 뜻이면 상황이 달라질 수 있다.

따라서 단순히 `drive`만 맞다고 보지 말고, 최종 답변과 `think`를 분리해서 모델이 목표 객체인 차를 끝까지 유지하는지와 애매함을 잘 다루는지도 같이 보세요. `carwash_reasoning_diagnostics.csv`에서 답은 맞지만 thinking이 `walk_only_trap`이나 `missing_car_target`으로 잡히는 케이스를 우선 보면 됩니다.

참고한 현재 모델 후보:

- Qwen3.6 collection: `Qwen/Qwen3.6-27B`, `Qwen/Qwen3.6-35B-A3B` 및 FP8 변형
- Qwen3.5 collection: `Qwen/Qwen3.5-2B`, `Qwen/Qwen3.5-9B`, `Qwen/Qwen3.5-27B` 등
- LoopLM/Ouro: `ByteDance/Ouro-1.4B`
- Parcae looped models: `SandyResearch/parcae-770m` 등
